In [1]:

import sys
import os
import tifffile
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import SimpleITK as sitk
sys.path.append('/home/cyf/wbi/Virginia/code_for_paper/wholistic_registration/src/wholistic_registration')
current_dir = os.path.dirname(os.path.abspath("__file__"))
root_dir = os.path.dirname(current_dir)
sys.path.append(root_dir)

root_dir = Path('/home/cyf/wbi/Virginia/code_for_paper/data/simulation_v0420/repos')
from utils import simulation
# root_dir = Path('/home/cyf/wbi/Virginia/code/src/wholistic_registration/exp/simulation/repos')
zslices = [2, 10, 18]  # 选择 z slice
experiment_groups = ['art_R', 'amp', 'noise_level']  # 对应三组实验
results = {}  # 保存 MSE
results = {group: {} for group in experiment_groups}
repo_dirs = sorted([d for d in root_dir.iterdir() if d.is_dir() and d.name.startswith('rep')])
num_repos = len(repo_dirs)
print(f"Found {num_repos} repos: {[d.name for d in repo_dirs]}")

CuPy not available - falling back to NumPy (CPU only)
CuPy not available - import standardscipy
Found 10 repos: ['rep01', 'rep02', 'rep03', 'rep04', 'rep05', 'rep06', 'rep07', 'rep08', 'rep09', 'rep10']


In [2]:
import numpy as np
import SimpleITK as sitk


def run_bspline_same_size(
    fixed_np,
    moving_np,
    spacing=(1.0, 1.0, 1.0),   # (sy, sx, sz) in your numpy convention
    mesh_size=(6, 6, 1),       # initial B-spline control grid
    optimizer="lbfgsb",        # "lbfgsb" or "gradient_descent"
    metric="mean_squares",     # "mean_squares" or "correlation"
    sampling_percentage=0.2,
    shrink_factors=(4, 2, 1),
    smoothing_sigmas=(2, 1, 0),
    bspline_scale_factors=(1, 2, 4),
    gd_learning_rate=1.0,
    gd_iterations=100,
    default_pixel_value=None,
    verbose=False,
):
    """
    Same-size B-spline registration.

    Parameters
    ----------
    fixed_np : np.ndarray
        Shape (Y, X, Z)
    moving_np : np.ndarray
        Shape (Y, X, Z)
    spacing : tuple
        (sy, sx, sz)

    Returns
    -------
    disp_np_yxz : np.ndarray
        Shape (Y, X, Z, 3), channel order [y, x, z]
    warped_np_yxz : np.ndarray
        Shape (Y, X, Z)
    out_tx : sitk.Transform
        Final B-spline transform
    """
    fixed_np = np.asarray(fixed_np, dtype=np.float32)
    moving_np = np.asarray(moving_np, dtype=np.float32)

    assert fixed_np.shape == moving_np.shape, "fixed and moving must have the same shape"
    assert fixed_np.ndim == 3, f"Expected 3D input, got ndim={fixed_np.ndim}"

    sy, sx, sz = spacing

    # Your numpy is (Y, X, Z) -> convert to SITK array order (Z, Y, X)
    fixed = sitk.GetImageFromArray(np.transpose(fixed_np, (2, 0, 1)))
    moving = sitk.GetImageFromArray(np.transpose(moving_np, (2, 0, 1)))

    # SITK spacing order is (x, y, z)
    fixed.SetSpacing((sx, sy, sz))
    moving.SetSpacing((sx, sy, sz))

    # Initial B-spline transform
    initial_tx = sitk.BSplineTransformInitializer(
        image1=fixed,
        transformDomainMeshSize=list(mesh_size),
        order=3
    )

    R = sitk.ImageRegistrationMethod()
    
    # Metric
    if metric == "mean_squares":
        R.SetMetricAsMeanSquares()
    elif metric == "correlation":
        R.SetMetricAsCorrelation()
    else:
        raise ValueError(f"Unknown metric: {metric}")
    R.MetricUseFixedImageGradientFilterOff()
    R.MetricUseMovingImageGradientFilterOff()
    # Sampling
    if sampling_percentage is not None and sampling_percentage < 1.0:
        R.SetMetricSamplingStrategy(R.RANDOM)
        R.SetMetricSamplingPercentage(float(sampling_percentage))
    else:
        R.SetMetricSamplingStrategy(R.NONE)

    # Interpolator
    R.SetInterpolator(sitk.sitkLinear)

    # Multi-resolution image pyramid
    R.SetShrinkFactorsPerLevel(shrinkFactors=list(shrink_factors))
    R.SetSmoothingSigmasPerLevel(smoothingSigmas=list(smoothing_sigmas))
    R.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()

    # Multi-resolution B-spline refinement
    R.SetInitialTransformAsBSpline(
        initial_tx,
        inPlace=False,
        scaleFactors=list(bspline_scale_factors)
    )

    # Optimizer
    if optimizer == "lbfgsb":
        R.SetOptimizerAsLBFGSB(
            gradientConvergenceTolerance=1e-5,
            numberOfIterations=int(gd_iterations),
            maximumNumberOfCorrections=5,
            maximumNumberOfFunctionEvaluations=1000,
            costFunctionConvergenceFactor=1e7,
        )
    elif optimizer == "gradient_descent":
        R.SetOptimizerAsGradientDescent(
            learningRate=float(gd_learning_rate),
            numberOfIterations=int(gd_iterations),
            convergenceMinimumValue=1e-6,
            convergenceWindowSize=10,
        )
        R.SetOptimizerScalesFromPhysicalShift()
    else:
        raise ValueError(f"Unknown optimizer: {optimizer}")

    if verbose:
        def _print_iter():
            print(
                f"Iter={R.GetOptimizerIteration():03d}, "
                f"Metric={R.GetMetricValue():.6f}"
            )
        R.AddCommand(sitk.sitkIterationEvent, _print_iter)

    # Execute registration
    out_tx = R.Execute(fixed, moving)

    # Convert transform to dense displacement field on fixed grid
    displacement_field = sitk.TransformToDisplacementField(
        out_tx,
        sitk.sitkVectorFloat64,
        fixed.GetSize(),
        fixed.GetOrigin(),
        fixed.GetSpacing(),
        fixed.GetDirection()
    )

    # Resample moving to fixed
    if default_pixel_value is None:
        default_pixel_value = float(np.min(moving_np))

    motion_corrected_image = sitk.Resample(
        moving,
        fixed,
        out_tx,
        sitk.sitkBSpline,   # keep this consistent with your previous working pipeline
        float(default_pixel_value),
        moving.GetPixelID()
    )

    # Robust extraction: split vector image into 3 scalar channels
    disp_x_zyx = sitk.GetArrayFromImage(sitk.VectorIndexSelectionCast(displacement_field, 0))
    disp_y_zyx = sitk.GetArrayFromImage(sitk.VectorIndexSelectionCast(displacement_field, 1))
    disp_z_zyx = sitk.GetArrayFromImage(sitk.VectorIndexSelectionCast(displacement_field, 2))

    # (Z, Y, X) -> (Y, X, Z)
    disp_x_yxz = np.transpose(disp_x_zyx, (1, 2, 0)).astype(np.float32)
    disp_y_yxz = np.transpose(disp_y_zyx, (1, 2, 0)).astype(np.float32)
    disp_z_yxz = np.transpose(disp_z_zyx, (1, 2, 0)).astype(np.float32)

    # SITK displacement channel order is [x, y, z]
    # convert to your convention [y, x, z]
    disp_np_yxz = np.stack([disp_y_yxz, disp_x_yxz, disp_z_yxz], axis=-1)

    warped_zyx = sitk.GetArrayFromImage(motion_corrected_image)
    warped_np_yxz = np.transpose(warped_zyx, (1, 2, 0)).astype(np.float32)

    return disp_np_yxz, warped_np_yxz, out_tx

In [ ]:
import os
import time
import numpy as np
import tifffile
import SimpleITK as sitk
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

# -------------------------
# 并行配置
# -------------------------
TOTAL_CPUS = os.cpu_count() or 8
THREADS_PER_CASE = 4
MAX_WORKERS = 32

print(f"[Parallel Config] TOTAL_CPUS={TOTAL_CPUS}, THREADS_PER_CASE={THREADS_PER_CASE}, MAX_WORKERS={MAX_WORKERS}")

# -------------------------
# bspline 参数
# -------------------------
bspline_params = {
    "mesh_size": (6, 6, 1),
    "optimizer": "lbfgsb",          # "lbfgsb" or "gradient_descent"
    "metric": "mean_squares",       # "mean_squares" or "correlation"
    "sampling_percentage": 0.2,
    "shrink_factors": (1,),
    "smoothing_sigmas": (0,),
    "bspline_scale_factors": (1,),
    "gd_learning_rate": 1.0,
    "gd_iterations": 100,
    "default_pixel_value": None,
    "verbose": False,
}

crop = 50
z_ratio_eval = 3.0769230769230766 * 8
bspline_spacing = (1.0, 1.0, 3.0769230769230766 * 8)


def process_one_case(repo_dir_str, group, exp_level, zslices, bspline_params, crop, z_ratio_eval, bspline_spacing, threads_per_case):
    """
    Process one experiment case.
    Return:
        (group, exp_level, metrics_dict)
    """
    sitk.ProcessObject.SetGlobalDefaultNumberOfThreads(int(threads_per_case))

    repo_dir = Path(repo_dir_str)
    subfolder = repo_dir / group / exp_level

    ref_path = subfolder / 'ref.tif'
    mov_path = subfolder / 'mov.tif'
    motion_path = subfolder / 'motion.tif'

    metrics = {
        'bspline_intensity_mse': np.nan,
        'bspline_intensity_rmse': np.nan,
        'bspline_intensity_ssim': np.nan,
        'bspline_runtime_sec': np.nan,
        'bspline_motion_epe_xy_mean': np.nan,
        'bspline_motion_epe_xy_median': np.nan,
        'bspline_motion_rmse_xy': np.nan,
        'bspline_motion_mae_y': np.nan,
        'bspline_motion_mae_x': np.nan,
        'bspline_motion_rmse_y': np.nan,
        'bspline_motion_rmse_x': np.nan,
        'bspline_motion_epe_3d_mean': np.nan,
        'bspline_motion_epe_3d_median': np.nan,
        'bspline_motion_epe_phys_mean': np.nan,
        'bspline_motion_epe_phys_median': np.nan,
        'bspline_motion_rmse_z': np.nan,
        'bspline_motion_mae_z': np.nan,
    }

    try:
        if not ref_path.exists() or not mov_path.exists() or not motion_path.exists():
            return group, exp_level, metrics

        ref_data = tifffile.imread(ref_path)[..., 2:19]
        move_data = tifffile.imread(mov_path)[..., zslices]
        motion_data = tifffile.imread(motion_path)[:, :, zslices, :]

        ref_data_clip = ref_data[:, :, [0, 8, 16]]

        mov_eval = simulation.crop_valid(move_data, crop=crop)
        gt_motion_eval = simulation.crop_valid(motion_data, crop=crop)

        t0 = time.perf_counter()
        motion_bspline, data_mov_bspline, _ = run_bspline_same_size(
            fixed_np=move_data,
            moving_np=ref_data_clip,
            spacing=bspline_spacing,
            mesh_size=bspline_params["mesh_size"],
            optimizer=bspline_params["optimizer"],
            metric=bspline_params["metric"],
            sampling_percentage=bspline_params["sampling_percentage"],
            shrink_factors=bspline_params["shrink_factors"],
            smoothing_sigmas=bspline_params["smoothing_sigmas"],
            bspline_scale_factors=bspline_params["bspline_scale_factors"],
            gd_learning_rate=bspline_params["gd_learning_rate"],
            gd_iterations=bspline_params["gd_iterations"],
            default_pixel_value=bspline_params["default_pixel_value"],
            verbose=bspline_params["verbose"],
        )
        bspline_runtime_sec = time.perf_counter() - t0

        motion_bspline_eval = simulation.crop_valid(motion_bspline, crop=crop)
        bspline_img_eval = simulation.crop_valid(data_mov_bspline, crop=crop)
        gt_motion_eval = simulation.crop_valid(motion_data, crop=crop)

        metrics['bspline_intensity_mse'] = simulation.compute_intensity_mse(bspline_img_eval, mov_eval)
        metrics['bspline_intensity_rmse'] = simulation.compute_intensity_rmse(bspline_img_eval, mov_eval)
        metrics['bspline_intensity_ssim'] = simulation.compute_volume_ssim(bspline_img_eval, mov_eval)
        metrics['bspline_runtime_sec'] = bspline_runtime_sec

        bspline_motion_metrics = simulation.compute_motion_metrics(
            motion_bspline_eval,
            gt_motion_eval,
            z_ratio=z_ratio_eval,
            use_z=True
        )

        metrics['bspline_motion_epe_xy_mean'] = bspline_motion_metrics['motion_epe_xy_mean']
        metrics['bspline_motion_epe_xy_median'] = bspline_motion_metrics['motion_epe_xy_median']
        metrics['bspline_motion_rmse_xy'] = bspline_motion_metrics['motion_rmse_xy']
        metrics['bspline_motion_mae_y'] = bspline_motion_metrics['motion_mae_y']
        metrics['bspline_motion_mae_x'] = bspline_motion_metrics['motion_mae_x']
        metrics['bspline_motion_rmse_y'] = bspline_motion_metrics['motion_rmse_y']
        metrics['bspline_motion_rmse_x'] = bspline_motion_metrics['motion_rmse_x']

        if 'motion_epe_3d_mean' in bspline_motion_metrics:
            metrics['bspline_motion_epe_3d_mean'] = bspline_motion_metrics['motion_epe_3d_mean']
            metrics['bspline_motion_epe_3d_median'] = bspline_motion_metrics['motion_epe_3d_median']
            metrics['bspline_motion_epe_phys_mean'] = bspline_motion_metrics['motion_epe_phys_mean']
            metrics['bspline_motion_epe_phys_median'] = bspline_motion_metrics['motion_epe_phys_median']
            metrics['bspline_motion_rmse_z'] = bspline_motion_metrics['motion_rmse_z']
            metrics['bspline_motion_mae_z'] = bspline_motion_metrics['motion_mae_z']

        print(
            f"[DONE] {repo_dir.name}/{group}/{exp_level} | "
            f"MSE={metrics['bspline_intensity_mse']:.4f}, "
            f"EPE_xy={metrics['bspline_motion_epe_xy_mean']:.4f}, "
            f"time={metrics['bspline_runtime_sec']:.2f}s"
        )

    except Exception as e:
        print(f"[FAILED] {repo_dir.name}/{group}/{exp_level}: {e}")

    return group, exp_level, metrics


# -------------------------
# 收集所有任务
# -------------------------
tasks = []
for repo_dir in repo_dirs:
    for group in experiment_groups:
        group_dir = repo_dir / group
        if not group_dir.exists():
            continue
        for subfolder in sorted(group_dir.iterdir(), key=lambda x: int(x.name)):
            if not subfolder.is_dir():
                continue
            exp_level = subfolder.name
            tasks.append((str(repo_dir), group, exp_level))

print(f"[Task Count] {len(tasks)} cases")

# -------------------------
# 并行执行
# -------------------------
results = {group: {} for group in experiment_groups}

with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [
        executor.submit(
            process_one_case,
            repo_dir_str,
            group,
            exp_level,
            zslices,
            bspline_params,
            crop,
            z_ratio_eval,
            bspline_spacing,
            THREADS_PER_CASE
        )
        for repo_dir_str, group, exp_level in tasks
    ]

    for future in as_completed(futures):
        group, exp_level, case_metrics = future.result()

        if exp_level not in results[group]:
            results[group][exp_level] = {
                'bspline_intensity_mse': [],
                'bspline_intensity_rmse': [],
                'bspline_intensity_ssim': [],
                'bspline_runtime_sec': [],
                'bspline_motion_epe_xy_mean': [],
                'bspline_motion_epe_xy_median': [],
                'bspline_motion_rmse_xy': [],
                'bspline_motion_mae_y': [],
                'bspline_motion_mae_x': [],
                'bspline_motion_rmse_y': [],
                'bspline_motion_rmse_x': [],
                'bspline_motion_epe_3d_mean': [],
                'bspline_motion_epe_3d_median': [],
                'bspline_motion_epe_phys_mean': [],
                'bspline_motion_epe_phys_median': [],
                'bspline_motion_rmse_z': [],
                'bspline_motion_mae_z': [],
            }

        for k, v in case_metrics.items():
            results[group][exp_level][k].append(v)

# -------------------------
# processed_results 自动汇总
# -------------------------
processed_results = {}

for group in experiment_groups:
    if group not in results or not results[group]:
        continue

    sorted_exp_levels = sorted(results[group].keys(), key=int)

    processed_results[group] = {
        'labels': sorted_exp_levels
    }

    metric_names = set()
    for lv in sorted_exp_levels:
        metric_names.update(results[group][lv].keys())

    for metric_name in sorted(metric_names):
        processed_results[group][f'avg_{metric_name}'] = [
            float(np.nanmean(results[group][lv][metric_name])) if len(results[group][lv][metric_name]) > 0 else np.nan
            for lv in sorted_exp_levels
        ]
        processed_results[group][f'std_{metric_name}'] = [
            float(np.nanstd(results[group][lv][metric_name])) if len(results[group][lv][metric_name]) > 0 else np.nan
            for lv in sorted_exp_levels
        ]

# -------------------------
# 保存
# -------------------------
output_file = "/home/cyf/wbi/Virginia/code_for_paper/Fig2/2.1_simulation/result/BSpline.npy"

np.save(output_file, {
    'raw_results': results,
    'processed_results': processed_results,
    'bspline_params': bspline_params,
    'parallel_config': {
        'total_cpus': TOTAL_CPUS,
        'threads_per_case': THREADS_PER_CASE,
        'max_workers': MAX_WORKERS,
    }
}, allow_pickle=True)

print(f"Saved all results to {output_file}")

[Parallel Config] TOTAL_CPUS=256, THREADS_PER_CASE=4, MAX_WORKERS=32
[Task Count] 290 cases


[DONE] rep01/noise_level/8 | MSE=1443.1306, EPE_xy=0.8621, time=311.79s
[DONE] rep01/noise_level/9 | MSE=1528.6743, EPE_xy=1.3826, time=315.18s
[DONE] rep02/art_R/3 | MSE=1846.1505, EPE_xy=3.6223, time=315.11s
[DONE] rep01/noise_level/4 | MSE=1405.6466, EPE_xy=0.8644, time=316.54s
[DONE] rep01/art_R/6 | MSE=1532.9130, EPE_xy=2.7469, time=318.02s
[DONE] rep02/art_R/1 | MSE=2134.2700, EPE_xy=4.2872, time=318.42s
[DONE] rep01/art_R/1 | MSE=2025.4060, EPE_xy=4.8235, time=319.20s
[DONE] rep01/amp/9 | MSE=2222.4092, EPE_xy=3.6952, time=323.75s
[DONE] rep01/art_R/5 | MSE=1724.3153, EPE_xy=2.8370, time=324.16s
[DONE] rep01/amp/6 | MSE=1530.4667, EPE_xy=2.4603, time=324.42s
[DONE] rep01/noise_level/6 | MSE=1381.4291, EPE_xy=1.1942, time=324.86s
[DONE] rep01/amp/2 | MSE=1624.4320, EPE_xy=1.1675, time=325.59s
[DONE] rep01/art_R/4 | MSE=1543.9342, EPE_xy=3.1259, time=325.64s
[DONE] rep01/amp/7 | MSE=2502.9368, EPE_xy=2.4609, time=326.76s
[DONE] rep01/noise_level/5 | MSE=1272.6614, EPE_xy=1.2764, t